# ChemoMAE on real NIR hyperspectral mineral specimens

**Unexecuted development draft.** This notebook uses the openly licensed [Minerals in the Wild dataset](https://github.com/EleftheriaTtl/minerals-in-the-wild/blob/main/README.md), [release v1.0.0](https://github.com/EleftheriaTtl/minerals-in-the-wild/releases/tag/v1.0.0), DOI [10.5281/zenodo.21414930](https://doi.org/10.5281/zenodo.21414930). Attribute the data to **E. Tetoula-Tsonga, G. Arvanitakis, and T. Giannakas**, under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The SWIR (short-wave infrared) reflectance cubes have shape `(height, width, 273)`, span 996.34–2504.28 nm, and have individual specimen IDs and all-band-NaN background. Each valid pixel supplies one spectrum, linearly interpolated to 256 bands on the wavelength axis before SNV. The separate specimen-level XRF table is not loaded or SNV-normalized.

This draft targets ChemoMAE v0.2.3. Download behavior, new source publication, and fresh-Colab **Run all** are pending verification. Installation currently requests the moving ChemoMAE `main` ref; a release tutorial must pin both the matching package source and wavelength-metadata ref. No execution outputs or performance/quality results are supplied here.

The quick path reads specimens `sample_0` through `sample_11` from **part01 only**, using an 8/2/2 specimen holdout, at most 128 sampled pixels per specimen, two training epochs, and at most 48×48 native-coordinate crops of the two held-out specimens. **The archive download itself is not a twelve-specimen download:** its actual transfer size is printed, with a hard 1600 MiB cap. No Google Drive is required.

These constants illustrate library usage. The first twelve specimens are not a representative benchmark. Site/context grouping and related acquisitions are unverified, so this is specimen holdout only. Cluster IDs are learned groups, not mineral classes. Specimen XRF measurements are not pixel annotations and are not used as pixel ground truth.

## Feature-to-cell coverage

| Feature | Cell group |
| --- | --- |
| Install/source/API preflight, CPU and opt-in CUDA | 1 |
| Capped release-checksum-verified download, recorded metadata | 2 |
| True specimen split, pixel indices, native spatial crops | 3 |
| 273→256 wavelength interpolation before public SNV | 3 |
| Bounded public FPS; TGN and fractional shift with owned RNG | 4 |
| Optimizer/scheduler, public Trainer hooks, epoch-boundary resume | 5 |
| Final raw/EMA choice, reload, all-visible full-spectrum Tester | 6 |
| CLS, raw latent, configured latent, normalized latent, streamed Extractor | 7 |
| Train-only cosine clustering, fixed holdout prediction, exact persistence | 8 |
| SNV spectral cluster baseline, real maps, LLA/occupancy/silhouette | 9 |
| Optional bounded vMF mixture | 10 |
| Protocol/config/RNG/index hashes, runtime/versions/outputs | 11 |

LFR perturbation repetitions and scientific model/K selection require a separately specified experimental protocol; the quick path does not implement or claim them.

## 1. Install, choose the runtime, and check public APIs

Run this notebook only after the new APIs are published at the requested ref. The preflight fails before data downloading if an older package is installed. Installation uses existing ChemoMAE dependencies; Torch is supplied by the Colab runtime.

In [ ]:
import time
_INSTALL_STARTED = time.perf_counter()
%pip install --quiet "git+https://github.com/Mantis-Ryuji/ChemoMAE.git@main"
_INSTALL_ELAPSED = time.perf_counter() - _INSTALL_STARTED


In [ ]:
import csv
import hashlib
import importlib.metadata
import inspect
import io
import json
import math
import platform
import random
import tempfile
import urllib.request
import uuid
import zipfile
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

import chemomae
try:
    from chemomae.models import ChemoMAE
    from chemomae.preprocessing import snv, cosine_fps_downsample
    from chemomae.training import (
        Trainer, TrainerConfig, Tester, TesterConfig, Extractor, ExtractorConfig,
        SpectraAugmenter, SpectraAugmenterConfig, build_optimizer, build_scheduler,
    )
    from chemomae.training.trainer import PreparedBatch
    from chemomae.clustering import (
        CosineKMeans, VMFMixture, local_label_agreement, silhouette_score_cosine_gpu,
    )
except ImportError as exc:
    raise RuntimeError(
        "The development tutorial requires published matching ChemoMAE APIs; pin/install that source first."
    ) from exc

required_trainer_hooks = (
    "prepare_batch", "forward_batch", "compute_loss", "train_batches",
    "before_epoch", "after_epoch", "before_step", "after_step",
    "checkpoint_extra_state", "load_checkpoint_extra_state",
)
missing = [name for name in required_trainer_hooks if not hasattr(Trainer, name)]
if not hasattr(ChemoMAE, "encode") or not hasattr(Extractor, "iter_transform"):
    missing.append("encode/iter_transform")
if "generator" not in inspect.signature(SpectraAugmenter.forward).parameters:
    missing.append("SpectraAugmenter.forward(generator=...)")
if "device" not in inspect.signature(cosine_fps_downsample).parameters:
    missing.append("cosine_fps_downsample(device=...)")
if not {"output_type", "output_device", "output_dtype", "representation"}.issubset(
    ExtractorConfig.__dataclass_fields__
):
    missing.append("Extractor output contract")
if missing:
    raise RuntimeError(
        "This unreleased notebook needs the new public APIs. Publish/pin the matching source first: "
        + ", ".join(missing)
    )

SEED = 42
USE_CUDA = False  # Explicit opt-in: set True to use an available CUDA runtime.
DEVICE = torch.device("cuda" if USE_CUDA and torch.cuda.is_available() else "cpu")
AMP = False
PIXELS_PER_SPECIMEN = 128
CROP_LIMIT = 48
BATCH_SIZE = 64
EPOCHS = 2
K = 4
KMEANS_MAX_ITER = 30
SILHOUETTE_LIMIT = 256
USE_EMA = True
SELECTED_WEIGHTS = "ema"  # Predetermined choice; no validation/test model selection.
RUN_OPTIONAL_VMF = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats(DEVICE)

PIPELINE_STARTED = time.perf_counter()
CACHE = Path(tempfile.gettempdir()) / "chemomae-nir-hsi-cache" / "v1.0.0"
CACHE.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="chemomae-nir-hsi-run-"))
MODEL_DIR = RUN_DIR / "training"
RELEASE = "v1.0.0"
RELEASE_BASE = (
    "https://github.com/EleftheriaTtl/minerals-in-the-wild/releases/download/" + RELEASE
)
ARCHIVE_NAME = "hsi_cubes_part01.zip"
WAVELENGTH_URL = (
    "https://raw.githubusercontent.com/EleftheriaTtl/minerals-in-the-wild/"
    "main/metadata/wavelengths.csv"
)
DOWNLOAD_CAP = 1600 * 1024**2
MAX_CUBE_BYTES = 128 * 1024**2
transfer_records = []

dist = importlib.metadata.distribution("chemomae")
direct_url_text = dist.read_text("direct_url.json")
source_provenance = json.loads(direct_url_text) if direct_url_text else None
versions = {
    "python": platform.python_version(), "platform": platform.platform(),
    "chemomae": chemomae.__version__, "torch": torch.__version__,
    "numpy": np.__version__, "matplotlib": importlib.metadata.version("matplotlib"),
    "scipy": importlib.metadata.version("scipy"),
    "source_install": source_provenance,
}
print("Runtime:", DEVICE, "AMP:", AMP, "run directory:", RUN_DIR)
print("Source provenance:", json.dumps(source_provenance, indent=2))
print("CUDA available:", torch.cuda.is_available(), "(use is opt-in)")


## 2. Download part01 with a byte cap and verify its checksum

The release contains four archives, a checksum file, and an asset manifest. Only part01 is requested. The creator's release script uses numeric specimen order and basename `sample_N.npy` entries; the cell below verifies the twelve expected entries before reading them.

Dataset bytes stay in the external runtime cache. The downloader streams to a temporary file, enforces the cap even without a Content-Length header, and verifies the archive against the release's SHA256SUMS. Metadata are saved and hashed as downloaded. The moving `main` wavelength ref is a development limitation; recording its hash does not replace publication-time ref pinning.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def fetch_to_cache(url, destination, *, cap, expected_sha256=None):
    destination = Path(destination)
    if destination.exists():
        if destination.stat().st_size > cap:
            raise ValueError(f"Cached file exceeds the cap: {destination}")
        actual_hash = sha256_file(destination)
        if expected_sha256 is not None and actual_hash != expected_sha256:
            raise ValueError(f"Cached checksum mismatch; inspect/remove that cache file: {destination}")
        record = {
            "requested_url": url, "cache_hit": True,
            "actual_bytes": destination.stat().st_size, "sha256": actual_hash,
        }
        transfer_records.append(record)
        return record
    partial = destination.with_name(destination.name + ".partial-" + uuid.uuid4().hex)
    digest = hashlib.sha256()
    received = 0
    started = time.perf_counter()
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "ChemoMAE-development-tutorial"})
        with urllib.request.urlopen(request, timeout=120) as response, partial.open("wb") as sink:
            length_header = response.headers.get("Content-Length")
            declared = int(length_header) if length_header is not None else None
            if declared is not None and declared > cap:
                raise ValueError(f"Declared transfer size {declared} exceeds {cap} bytes.")
            final_url = response.geturl()
            etag = response.headers.get("ETag")
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                received += len(block)
                if received > cap:
                    raise ValueError(f"Transfer exceeded the hard cap of {cap} bytes.")
                sink.write(block)
                digest.update(block)
        actual_hash = digest.hexdigest()
        if expected_sha256 is not None and actual_hash != expected_sha256:
            raise ValueError(f"Release checksum mismatch for {destination.name}.")
        partial.replace(destination)
    except Exception:
        partial.unlink(missing_ok=True)
        raise
    record = {
        "requested_url": url, "resolved_url": final_url, "etag": etag,
        "cache_hit": False, "declared_bytes": declared, "actual_bytes": received,
        "sha256": actual_hash, "elapsed_seconds": time.perf_counter() - started,
    }
    transfer_records.append(record)
    return record

checksum_path = CACHE / "SHA256SUMS"
asset_manifest_path = CACHE / "asset_manifest.csv"
wavelength_path = CACHE / "wavelengths-main.csv"
fetch_to_cache(RELEASE_BASE + "/SHA256SUMS", checksum_path, cap=2 * 1024**2)
fetch_to_cache(RELEASE_BASE + "/asset_manifest.csv", asset_manifest_path, cap=2 * 1024**2)
fetch_to_cache(WAVELENGTH_URL, wavelength_path, cap=1024**2)
checksums = {}
for line in checksum_path.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    parts = line.split(maxsplit=1)
    if len(parts) != 2 or len(parts[0]) != 64 or any(c not in "0123456789abcdefABCDEF" for c in parts[0]):
        raise ValueError("Unrecognized SHA256SUMS entry.")
    checksums[Path(parts[1].lstrip("*").strip()).name] = parts[0].lower()
if ARCHIVE_NAME not in checksums:
    raise ValueError(f"{ARCHIVE_NAME} is missing from the release checksums.")
archive_path = CACHE / ARCHIVE_NAME
archive_transfer = fetch_to_cache(
    RELEASE_BASE + "/" + ARCHIVE_NAME, archive_path,
    cap=DOWNLOAD_CAP, expected_sha256=checksums[ARCHIVE_NAME],
)
print("Archive bytes:", archive_transfer["actual_bytes"])
print("Archive MiB:", archive_transfer["actual_bytes"] / 1024**2)
print("Archive is not a tiny subset transfer. Hard cap MiB:", DOWNLOAD_CAP / 1024**2)
print("Checksum:", archive_transfer["sha256"])

with asset_manifest_path.open(encoding="utf-8-sig", newline="") as stream:
    asset_reader = csv.DictReader(stream)
    asset_manifest_columns = asset_reader.fieldnames
    asset_manifest_preview = list(asset_reader)[:8]
print("Asset manifest columns:", asset_manifest_columns)

with wavelength_path.open(encoding="utf-8-sig", newline="") as stream:
    wavelength_rows = list(csv.DictReader(stream))
required_columns = {"band_index", "wavelength_nm", "fwhm_nm"}
if len(wavelength_rows) != 273 or not required_columns.issubset(wavelength_rows[0]):
    raise ValueError("Expected 273 recorded wavelength rows with the documented columns.")
band_indices = np.array([int(row["band_index"]) for row in wavelength_rows], dtype=np.int64)
original_grid = np.array([float(row["wavelength_nm"]) for row in wavelength_rows], dtype=np.float64)
if not np.isfinite(original_grid).all() or not np.all(np.diff(original_grid) > 0):
    raise ValueError("Recorded wavelengths must be finite and strictly increasing; no implicit reordering.")
if not np.all(np.diff(band_indices) == 1):
    raise ValueError("Band indices must follow consecutive recorded cube-band order.")
target_grid = np.linspace(original_grid[0], original_grid[-1], 256, dtype=np.float64)
print("Recorded coverage (nm):", original_grid[0], original_grid[-1])
print("Recorded bands -> model bands:", original_grid.size, "->", target_grid.size)


## 3. Split specimens, interpolate 273→256, and apply SNV

Use the recorded increasing wavelength grid and a uniform 256-point grid over the same endpoints. Linear interpolation occurs **before** SNV, with no extrapolation. Keep both grids, the metadata snapshot, and the method in the protocol manifest. This is a user-selected resampling choice; it does not create additional spectral information.

The fixed seed permutes twelve known specimen IDs into 8 training, 2 validation, and 2 held-out specimens. Pixels are sampled independently within each specimen and recorded by native row/column indices. All-band NaN is background; partially nonfinite spectra and infinities are rejected rather than silently relabeled. No labels or XRF targets enter fitting.

Held-out map crops are at most 48×48 real pixels. A crop is centered near a valid pixel nearest the original image center and clipped to the native image extent; smaller images stay smaller. Crop offsets, masks, and absolute coordinates are saved. No artificial grid is built from shuffled sampled spectra.

In [ ]:
candidate_ids = [f"sample_{i}" for i in range(12)]
split_rng = np.random.default_rng(SEED)
permuted = [candidate_ids[int(index)] for index in split_rng.permutation(len(candidate_ids))]
split_ids = {
    "train": permuted[:8], "validation": permuted[8:10], "heldout": permuted[10:],
}
assert len(set(sum(split_ids.values(), []))) == 12
role_by_id = {specimen: role for role, specimens in split_ids.items() for specimen in specimens}
pixel_indices = {}
selected_snv = {}
cube_records = []
heldout_crops = {}

# Precompute the same wavelength interpolation weights for every selected pixel.
left = np.searchsorted(original_grid, target_grid, side="right") - 1
left = np.clip(left, 0, original_grid.size - 2)
right = left + 1
alpha = (target_grid - original_grid[left]) / (original_grid[right] - original_grid[left])
if target_grid[0] < original_grid[0] or target_grid[-1] > original_grid[-1]:
    raise ValueError("Extrapolation is forbidden.")

def interpolate_spectra(spectra):
    spectra = np.asarray(spectra)
    if spectra.ndim != 2 or spectra.shape[1] != 273 or not np.isfinite(spectra).all():
        raise ValueError("Interpolation requires finite rows with 273 bands.")
    return ((1.0 - alpha) * spectra[:, left] + alpha * spectra[:, right]).astype(np.float32)

def array_sha256(array):
    contiguous = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(str(contiguous.dtype).encode())
    digest.update(json.dumps(list(contiguous.shape)).encode())
    digest.update(contiguous.tobytes())
    return digest.hexdigest()

def crop_near_valid_center(cube, valid):
    height, width = valid.shape
    coordinates = np.argwhere(valid)
    if coordinates.size == 0:
        raise ValueError("A specimen has no valid spatial pixels.")
    image_center = np.array([(height - 1) / 2, (width - 1) / 2])
    center_y, center_x = coordinates[np.argmin(((coordinates - image_center) ** 2).sum(axis=1))]
    crop_h, crop_w = min(CROP_LIMIT, height), min(CROP_LIMIT, width)
    y0 = int(np.clip(center_y - crop_h // 2, 0, height - crop_h))
    x0 = int(np.clip(center_x - crop_w // 2, 0, width - crop_w))
    ys, xs = slice(y0, y0 + crop_h), slice(x0, x0 + crop_w)
    crop_mask = valid[ys, xs].copy()
    crop_cube = cube[ys, xs].copy()
    local_coordinates = np.argwhere(crop_mask)
    absolute_coordinates = local_coordinates + np.array([y0, x0])
    return {
        "cube": crop_cube, "mask": crop_mask,
        "local_coordinates": local_coordinates, "absolute_coordinates": absolute_coordinates,
        "bounds": {"y0": y0, "x0": x0, "height": crop_h, "width": crop_w},
    }

with zipfile.ZipFile(archive_path) as archive:
    infos = {info.filename: info for info in archive.infolist()}
    for specimen in candidate_ids:
        member = specimen + ".npy"
        if member not in infos or infos[member].is_dir():
            raise ValueError(f"Expected flat ZIP entry is absent: {member}")
        if infos[member].file_size > MAX_CUBE_BYTES:
            raise ValueError(f"Selected cube exceeds the per-cube read cap: {member}")
        with archive.open(member) as stream:
            cube = np.load(stream, allow_pickle=False)
        if cube.dtype != np.float32 or cube.ndim != 3 or cube.shape[-1] != 273:
            raise ValueError(f"Unexpected cube dtype/shape for {specimen}: {cube.dtype}, {cube.shape}")
        background = np.isnan(cube).all(axis=-1)
        finite = np.isfinite(cube).all(axis=-1)
        if np.any(~background & ~finite):
            raise ValueError(f"{specimen} contains partial nonfinite spectra or infinity.")
        valid = finite
        native_coordinates = np.argwhere(valid)
        if native_coordinates.size == 0:
            raise ValueError(f"{specimen} has no valid pixels.")
        specimen_number = int(specimen.removeprefix("sample_"))
        sample_rng = np.random.default_rng(SEED + 1000 + specimen_number)
        selected_positions = np.sort(sample_rng.choice(
            len(native_coordinates), size=min(PIXELS_PER_SPECIMEN, len(native_coordinates)), replace=False
        ))
        coordinates = native_coordinates[selected_positions]
        raw_spectra = cube[coordinates[:, 0], coordinates[:, 1]]
        interpolated = interpolate_spectra(raw_spectra)
        normalized = snv(torch.from_numpy(interpolated))
        pixel_indices[specimen] = coordinates.astype(np.int64)
        selected_snv[specimen] = normalized.cpu()
        cube_records.append({
            "specimen_id": specimen, "role": role_by_id[specimen],
            "member": member, "zip_crc32": int(infos[member].CRC),
            "zip_member_bytes": int(infos[member].file_size),
            "shape": list(cube.shape), "valid_pixels": int(valid.sum()),
            "background_pixels": int(background.sum()),
            "sampled_pixels": len(coordinates),
            "sample_index_sha256": array_sha256(pixel_indices[specimen]),
            "sample_index_seed": SEED + 1000 + specimen_number,
            "zero_snv_rows": int((normalized.norm(dim=1) == 0).sum()),
        })
        if role_by_id[specimen] == "heldout":
            heldout_crops[specimen] = crop_near_valid_center(cube, valid)
        if specimen == split_ids["train"][0]:
            example_raw = raw_spectra[0].copy()
            example_interpolated = interpolated[0].copy()
            example_snv = normalized[0].numpy().copy()
        del cube, background, finite, valid, raw_spectra, interpolated, normalized

train_x = torch.cat([selected_snv[s] for s in split_ids["train"]])
validation_x = torch.cat([selected_snv[s] for s in split_ids["validation"]])
heldout_x = torch.cat([selected_snv[s] for s in split_ids["heldout"]])
if len(train_x) < K:
    raise ValueError("Too few training pixels for the declared cluster count.")
index_manifest_path = RUN_DIR / "sample_indices.npz"
np.savez(index_manifest_path, **pixel_indices)
np.savez(RUN_DIR / "wavelength_grids.npz", original_nm=original_grid, target_nm=target_grid)
for specimen, crop in heldout_crops.items():
    np.savez(
        RUN_DIR / f"{specimen}_crop_geometry.npz", valid_mask=crop["mask"],
        local_coordinates=crop["local_coordinates"],
        absolute_coordinates=crop["absolute_coordinates"],
        **{key: np.array(value) for key, value in crop["bounds"].items()},
    )
print("Specimen split:", split_ids)
print("Sampled rows:", len(train_x), len(validation_x), len(heldout_x))
print("Crop geometry:", {s: c["bounds"] for s, c in heldout_crops.items()})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(original_grid, example_raw, label="recorded 273 bands", alpha=0.7)
axes[0].plot(target_grid, example_interpolated, label="linear 256 bands", linewidth=1)
axes[0].set(xlabel="Wavelength (nm)", ylabel="Supplied spectral signal", title="Training specimen example")
axes[0].legend()
axes[1].plot(target_grid, example_snv)
axes[1].set(xlabel="Wavelength (nm)", ylabel="SNV", title="Interpolation before per-spectrum SNV")
fig.tight_layout()
fig.savefig(RUN_DIR / "preprocessing_example.png", dpi=130)
plt.show()


## 4. Bounded FPS and explicit augmentation streams

FPS is demonstrated on at most 128 training pixels, selecting 16 directions; it does not replace this notebook's recorded random training subset. FS controls channel displacement, while TGN controls geodesic angle. Their illustrative strengths are not scientifically optimized here. Both augmentation and masking use independent device-matched generators; loading/shuffling uses its own CPU generator.

In [ ]:
fps_pool = train_x[:min(128, len(train_x))]
fps_target = min(16, len(fps_pool))
fps_subset, fps_indices = cosine_fps_downsample(
    fps_pool, ratio=fps_target / len(fps_pool), seed=SEED,
    device=DEVICE, return_numpy=False, return_indices=True,
)
fps_indices_cpu = fps_indices.cpu().numpy()
np.save(RUN_DIR / "fps_training_pool_indices.npy", fps_indices_cpu)
augmentation_config = SpectraAugmenterConfig(
    shift_prob=0.5, shift_delta_range=(-1.0, 1.0),
    noise_prob=0.5, noise_angle_deg_range=(0.5, 2.0),
    shuffle_order_per_batch=True, recenter_after_each_op=True, renorm_to_input_norm=True,
)
demo_stream = torch.Generator(device=DEVICE).manual_seed(SEED + 10)
demo_augmenter = SpectraAugmenter(augmentation_config, generator=demo_stream).train()
demo_clean = train_x[:4].to(DEVICE)
demo_augmented = demo_augmenter(demo_clean).cpu()
print("FPS pool/selected:", len(fps_pool), len(fps_subset), "compute device:", DEVICE)
print("Demonstration mean/norm shifts:",
      (demo_augmented.mean(dim=1) - demo_clean.cpu().mean(dim=1)).abs().max().item(),
      (demo_augmented.norm(dim=1) - demo_clean.cpu().norm(dim=1)).abs().max().item())


## 5. Use public Trainer hooks and resume at a completed epoch

Use a tiny 256-channel model with 16 equal patches, patch length 16, 8 hidden patches, width 32, one Transformer layer, and latent width 8. Move it to the chosen device **before** constructing its optimizer. AMP and TF32 stay off for this baseline.

The subclass below only prepares a public `PreparedBatch`, adds protocol metadata, and persists caller-owned RNG through the public checkpoint extension hooks. The library owns forward/loss/backward/clipping/optimizer/scheduler/EMA steps; no private training loop is copied. We stop after epoch 1, reconstruct the training objects, and resume to the absolute final epoch 2.

The saved boundary is the completed epoch, with single-process loading (`num_workers=0`). This does not claim restoration of arbitrary external workers, services, or mid-epoch position. A rigorous resume-equivalence check against an uninterrupted run remains a separate validation gate.

In [ ]:
model_config = {
    "seq_len": 256, "n_patches": 16, "n_mask": 8, "d_model": 32, "nhead": 4,
    "num_layers": 1, "dim_feedforward": 64, "dropout": 0.0,
    "latent_dim": 8, "latent_normalize": True, "decoder_num_layers": 2,
}
protocol = {
    "status": "illustrative_development_run",
    "dataset": {"name": "Minerals in the Wild", "release": RELEASE,
                "doi": "10.5281/zenodo.21414930", "license": "CC-BY-4.0",
                "attribution": ["E. Tetoula-Tsonga", "G. Arvanitakis", "T. Giannakas"],
                "archive": ARCHIVE_NAME, "archive_sha256": archive_transfer["sha256"],
                "candidate_specimens": candidate_ids, "site_grouping_verified": False,
                "pixel_annotations_available": False, "xrf_used": False},
    "split": split_ids, "split_unit": "individual_specimen", "seed": SEED,
    "random_streams": {"mask_seed": SEED + 101, "augmentation_seed": SEED + 202,
        "loader_seed": SEED + 303, "demo_augmentation_seed": SEED + 10,
        "pixel_sampling": "seed + 1000 + numeric specimen ID",
        "silhouette_sampling": "seed + 4000 + numeric specimen ID",
        "fps_seed": SEED, "cluster_initialization_seed": SEED,
        "loader_workers": 0, "deterministic_algorithms_enforced": False},
    "pixels_per_specimen_cap": PIXELS_PER_SPECIMEN, "cube_records": cube_records,
    "interpolation": {"method": "linear", "extrapolation": False,
                      "recorded_band_indices": band_indices.tolist(),
                      "recorded_wavelength_nm": original_grid.tolist(),
                      "target_wavelength_nm": target_grid.tolist(),
                      "metadata_url": WAVELENGTH_URL,
                      "metadata_sha256": sha256_file(wavelength_path)},
    "snv": {"implementation": "chemomae.preprocessing.snv", "eps": 1e-12,
            "statistics": "per-spectrum; library sample-std convention"},
    "model": model_config, "training": {"epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "amp": AMP, "tf32": False, "use_ema": USE_EMA, "ema_decay": 0.9,
        "selected_weights": SELECTED_WEIGHTS, "optimizer": "public build_optimizer",
        "lr": 1.5e-4, "weight_decay": 0.05, "warmup_epochs": 1,
        "selection_uses_validation_or_test": False},
    "augmentation": asdict(augmentation_config),
    "clustering": {"k": K, "max_iter": KMEANS_MAX_ITER, "seed": SEED,
                   "fit_data": "sampled training pixels only",
                   "representation": "normalized_latent"},
    "maps": {"crop_limit": CROP_LIMIT, "specimens": {
        s: {**crop["bounds"], "valid_mask_sha256": array_sha256(crop["mask"]),
            "absolute_coordinates_sha256": array_sha256(crop["absolute_coordinates"])}
        for s, crop in heldout_crops.items()
    }},
    "silhouette_cap": SILHOUETTE_LIMIT,
}
protocol_sha256 = hashlib.sha256(
    json.dumps(protocol, sort_keys=True, allow_nan=False).encode()
).hexdigest()
(RUN_DIR / "protocol.json").write_text(json.dumps(protocol, indent=2, allow_nan=False), encoding="utf-8")
(RUN_DIR / "model_config.json").write_text(json.dumps(model_config, indent=2), encoding="utf-8")

class TutorialTrainer(Trainer):
    def __init__(self, *args, mask_stream, augmentation_stream, loader_stream, protocol_hash, **kwargs):
        self.mask_stream = mask_stream
        self.augmentation_stream = augmentation_stream
        self.loader_stream = loader_stream
        self.protocol_hash = protocol_hash
        super().__init__(*args, **kwargs)

    def prepare_batch(self, batch):
        clean = batch[0].to(self.device, dtype=torch.float32)
        model_input = self.augmenter(clean) if self.augmenter is not None else clean
        visible = self.model.make_visible(
            len(clean), device=self.device, generator=self.mask_stream
        )
        return PreparedBatch(model_input=model_input, target=clean, visible_mask=visible)

    def after_epoch(self, epoch, record):
        record["protocol_sha256"] = self.protocol_hash
        record["training_specimens"] = len(split_ids["train"])

    def checkpoint_extra_state(self):
        return {
            "protocol_sha256": self.protocol_hash,
            "mask_stream": self.mask_stream.get_state().clone(),
            "augmentation_stream": self.augmentation_stream.get_state().clone(),
            "loader_stream": self.loader_stream.get_state().clone(),
            "torch_cpu_rng": torch.get_rng_state().clone(),
            "torch_cuda_rng": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else [],
            "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
        }

    def load_checkpoint_extra_state(self, state):
        if state.get("protocol_sha256") != self.protocol_hash:
            raise ValueError("Resume protocol mismatch.")
        self.mask_stream.set_state(state["mask_stream"])
        self.augmentation_stream.set_state(state["augmentation_stream"])
        self.loader_stream.set_state(state["loader_stream"])
        torch.set_rng_state(state["torch_cpu_rng"])
        if DEVICE.type == "cuda":
            torch.cuda.set_rng_state_all(state["torch_cuda_rng"])
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])

def make_training_objects(*, resume_from=None):
    mask_stream = torch.Generator(device=DEVICE).manual_seed(SEED + 101)
    augmentation_stream = torch.Generator(device=DEVICE).manual_seed(SEED + 202)
    loader_stream = torch.Generator(device="cpu").manual_seed(SEED + 303)
    train_loader = DataLoader(
        TensorDataset(train_x), batch_size=BATCH_SIZE, shuffle=True,
        generator=loader_stream, num_workers=0, drop_last=False,
    )
    model = ChemoMAE(**model_config).to(DEVICE)
    optimizer = build_optimizer(model, lr=1.5e-4, weight_decay=0.05)
    scheduler = build_scheduler(
        optimizer, steps_per_epoch=len(train_loader), epochs=EPOCHS,
        warmup_epochs=1, min_lr_scale=0.1,
    )
    augmenter = SpectraAugmenter(augmentation_config, generator=augmentation_stream)
    trainer = TutorialTrainer(
        model, optimizer, train_loader, scheduler=scheduler, augmenter=augmenter,
        cfg=TrainerConfig(
            out_dir=MODEL_DIR, device=str(DEVICE), amp=AMP, enable_tf32=False,
            use_ema=USE_EMA, ema_decay=0.9, loss_type="mse", loss_region="masked",
            reduction="mean", resume_from=resume_from,
        ),
        mask_stream=mask_stream, augmentation_stream=augmentation_stream,
        loader_stream=loader_stream, protocol_hash=protocol_sha256,
    )
    return model, optimizer, scheduler, trainer

model, optimizer, scheduler, first_trainer = make_training_objects()
optimizer_groups = [
    {"group": i, "parameters": sum(p.numel() for p in group["params"]),
     "lr_at_start": group["lr"], "weight_decay": group["weight_decay"]}
    for i, group in enumerate(optimizer.param_groups)
]
print("Inspectable optimizer groups:", optimizer_groups)
print("Scheduler advances after successful optimizer updates; AMP-skipped attempts do not advance EMA/scheduler.")
first_stage = first_trainer.fit(epochs=1)
print("Epoch-boundary checkpoint:", MODEL_DIR / "checkpoints" / "last.pt")
del first_trainer, optimizer, scheduler, model

model, optimizer, scheduler, resumed_trainer = make_training_objects(
    resume_from=MODEL_DIR / "checkpoints" / "last.pt"
)
training_result = resumed_trainer.fit(epochs=EPOCHS)
print("Cumulative result:", training_result)
torch.save(resumed_trainer.checkpoint_extra_state(), RUN_DIR / "final_rng_state.pt")


## 6. Reload the predetermined final export and evaluate clean reconstruction

Trainer exports final raw and, when enabled, final EMA weights. Its in-memory model is raw. Reload the chosen export explicitly before downstream work; `ema` here is a predetermined tutorial setting, not a validation-selected winner. Configuration is saved separately because architecture settings cannot all be recovered from weights.

Tester uses an all-visible fixed mask and `loss_region="all"`, computing full-spectrum MSE over the sampled validation/held-out pixels. Masked training loss and this all-visible evaluation answer different questions. Neither held-out set is used to fit centers, tune K, or select weights.

In [ ]:
if SELECTED_WEIGHTS not in {"raw", "ema"} or (SELECTED_WEIGHTS == "ema" and not USE_EMA):
    raise ValueError("Choose a valid predetermined export.")
selected_filename = "ema_last_model.pt" if SELECTED_WEIGHTS == "ema" else "last_model.pt"
selected_path = MODEL_DIR / selected_filename
saved_model_config = json.loads((RUN_DIR / "model_config.json").read_text(encoding="utf-8"))
inference_model = ChemoMAE(**saved_model_config).to(DEVICE)
inference_model.load_state_dict(torch.load(selected_path, map_location=DEVICE, weights_only=True), strict=True)
inference_model.eval()

def clean_loader(spectra):
    return DataLoader(TensorDataset(spectra), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

tester = Tester(
    inference_model,
    TesterConfig(
        device=DEVICE, amp=AMP, loss_type="mse", loss_region="all", reduction="mean",
        fixed_visible=torch.ones(256, dtype=torch.bool), log_history=False, progress=False,
    ),
)
reconstruction_metrics = {
    "validation_all_visible_full_mse": tester(clean_loader(validation_x)),
    "heldout_all_visible_full_mse": tester(clean_loader(heldout_x)),
    "validation_sampled_pixels": len(validation_x),
    "heldout_sampled_pixels": len(heldout_x),
    "selected_weights": SELECTED_WEIGHTS,
}
print("Selected export:", selected_path)
print("Clean reconstruction:", reconstruction_metrics)


## 7. Inspect all four representations and stream selected training features

`latent` follows the model's normalization flag; `raw_latent` precedes normalization, `normalized_latent` always normalizes, and `cls` precedes projection. All-visible `encode` leaves modes/gradients under caller control; Extractor handles inference scopes and restores them before each yield.

Stream training features to small caller-owned files, then concatenate the deliberately bounded training subset for clustering. That concatenation is our explicit memory choice; it is not mandatory inside the streaming API. Crop features below also remain bounded.

In [ ]:
representation_shapes = {}
representation_norms = {}
inspection_loader = clean_loader(train_x[:min(16, len(train_x))])
for representation in ("latent", "raw_latent", "normalized_latent", "cls"):
    inspected = Extractor(
        inference_model,
        ExtractorConfig(
            device=DEVICE, amp=AMP, representation=representation,
            output_type="tensor", output_device="cpu", output_dtype=torch.float32,
        ),
    ).transform(inspection_loader)
    representation_shapes[representation] = list(inspected.shape)
    representation_norms[representation] = {
        "minimum": inspected.norm(dim=1).min().item(),
        "maximum": inspected.norm(dim=1).max().item(),
    }

feature_extractor = Extractor(
    inference_model,
    ExtractorConfig(
        device=DEVICE, amp=AMP, representation="normalized_latent",
        output_type="tensor", output_device="cpu", output_dtype=torch.float32,
    ),
)
feature_dir = RUN_DIR / "training_feature_batches"
feature_dir.mkdir()
feature_batches = []
feature_batch_files = []
from contextlib import closing
with closing(feature_extractor.iter_transform(clean_loader(train_x))) as batches:
    for batch_number, features in enumerate(batches):
        feature_path = feature_dir / f"batch_{batch_number:04d}.npy"
        np.save(feature_path, features.numpy())
        feature_batch_files.append({
            "file": str(feature_path.relative_to(RUN_DIR)), "shape": list(features.shape),
            "sha256": sha256_file(feature_path),
        })
        feature_batches.append(features)
train_z = torch.cat(feature_batches)
del feature_batches

# Public PyTorch state/config reload, checked using the same small input batch.
roundtrip_model = ChemoMAE(**saved_model_config).to(DEVICE).eval()
roundtrip_model.load_state_dict(torch.load(selected_path, map_location=DEVICE, weights_only=True), strict=True)
probe = train_x[:min(8, len(train_x))].to(DEVICE)
with torch.no_grad():
    before_reload = inference_model.encode(probe, representation="normalized_latent")
    after_reload = roundtrip_model.encode(probe, representation="normalized_latent")
torch.testing.assert_close(before_reload, after_reload)
model_roundtrip_check = "assert_close completed in this runtime"
del roundtrip_model
print("Representation shapes:", representation_shapes)
print("Representation norms:", representation_norms)
print("Streamed training rows/features:", tuple(train_z.shape))


## 8. Fit only training features, freeze prediction, and reload exact centers

Use K=4 as a declared illustrative setting, with at most 30 centroid updates. Cluster identities are local numeric IDs. An SNV spectral baseline uses the same training pixels without a learned encoder. The two feature spaces have different dimensions and costs; this short run is not a fairness-controlled ranking.

Persist fitted state through `save_centroids`/`load_centroids`. Reloading must preserve centers exactly and produce the same predictions; it does not resume a centroid optimization or restore an advanced initialization stream.

In [ ]:
latent_clusterer = CosineKMeans(
    n_components=K, max_iter=KMEANS_MAX_ITER, device=DEVICE, random_state=SEED,
).fit(train_z, chunk=128)
spectral_baseline = CosineKMeans(
    n_components=K, max_iter=KMEANS_MAX_ITER, device=DEVICE, random_state=SEED,
).fit(train_x, chunk=128)

latent_cluster_path = RUN_DIR / "latent_centroids.pt"
baseline_cluster_path = RUN_DIR / "snv_centroids.pt"
latent_clusterer.save_centroids(latent_cluster_path)
spectral_baseline.save_centroids(baseline_cluster_path)
reloaded_clusterer = CosineKMeans(n_components=K, device=DEVICE, random_state=SEED)
reloaded_clusterer.load_centroids(latent_cluster_path)
assert torch.equal(latent_clusterer.centroids, reloaded_clusterer.centroids)
original_labels = latent_clusterer.predict(train_z, chunk=128)
reloaded_labels = reloaded_clusterer.predict(train_z, chunk=128)
assert torch.equal(original_labels, reloaded_labels)
cluster_roundtrip_check = "exact center and prediction assertions completed in this runtime"
latent_clusterer = reloaded_clusterer

clustering_diagnostics = {
    name: {"n_iter": fitted.n_iter_, "converged": fitted.converged_,
           "stop_reason": fitted.stop_reason_, "final_mean_cosine_dissimilarity": fitted.inertia_}
    for name, fitted in (("latent", latent_clusterer), ("snv_baseline", spectral_baseline))
}
print("Training fit diagnostics:", clustering_diagnostics)


## 9. Predict real held-out crops and inspect spatial agreement

Centers remain fixed. Every valid native crop pixel is interpolated and normalized, encoded, and assigned. All-band-NaN pixels remain excluded by an explicit mask; holes and boundaries are preserved. Cluster colors indicate numeric IDs only. No pixel mineral annotations are available.

LLA counts equal-label directed neighbor pairs with the center excluded and corrects for finite-sample class occupancy:

$$
A_w = \frac{a_w}{m_w}, \qquad P = \frac{\sum_k n_k(n_k-1)}{N(N-1)}, \qquad \mathrm{LLA}_w = \frac{A_w-P}{1-P}.
$$

Report raw agreement, corrected score, occupancy, pair counts, and explicit undefined reasons separately for each window. Single-class corrected scores are undefined; negative corrected scores are retained. LLA here describes **the crop**, excluding neighbors outside its edge, rather than the full specimen. High LLA does not establish chemical correctness or preserve small meaningful regions.

Cosine silhouette is computed on at most 256 held-out crop pixels with a recorded seed and a guard against undefined cluster counts. It describes those assigned features, not annotation agreement. Reconstruction error, spatial agreement, and silhouette measure different properties.

In [ ]:
map_metrics = {}
map_output_records = {}
for specimen, crop in heldout_crops.items():
    mask = crop["mask"]
    native_rows = crop["cube"][mask]
    crop_x = snv(torch.from_numpy(interpolate_spectra(native_rows)))
    crop_feature_batches = list(feature_extractor.iter_transform(clean_loader(crop_x)))
    crop_z = torch.cat(crop_feature_batches)
    del crop_feature_batches
    labels = latent_clusterer.predict(crop_z, chunk=128).cpu().numpy()
    baseline_labels = spectral_baseline.predict(crop_x, chunk=128).cpu().numpy()
    label_map = np.full(mask.shape, -1, dtype=np.int64)
    baseline_map = np.full(mask.shape, -1, dtype=np.int64)
    label_map[mask] = labels
    baseline_map[mask] = baseline_labels
    lla = local_label_agreement(label_map, mask, windows=(3, 5, 9), device=DEVICE, class_chunk=4)
    baseline_lla = local_label_agreement(baseline_map, mask, windows=(3, 5, 9), device=DEVICE, class_chunk=4)

    silhouette_rng = np.random.default_rng(SEED + 4000 + int(specimen.removeprefix("sample_")))
    metric_indices = np.sort(silhouette_rng.choice(
        len(labels), size=min(SILHOUETTE_LIMIT, len(labels)), replace=False
    ))
    metric_labels = labels[metric_indices]
    class_count = np.unique(metric_labels).size
    if 2 <= class_count < len(metric_indices):
        silhouette = silhouette_score_cosine_gpu(
            crop_z[metric_indices], metric_labels, device=str(DEVICE), chunk=128, dtype=torch.float32
        )
        silhouette_reason = None
    else:
        silhouette = None
        silhouette_reason = "silhouette_requires_2_to_N_minus_1_used_classes"
    baseline_metric_labels = baseline_labels[metric_indices]
    baseline_class_count = np.unique(baseline_metric_labels).size
    if 2 <= baseline_class_count < len(metric_indices):
        baseline_silhouette = silhouette_score_cosine_gpu(
            crop_x[metric_indices], baseline_metric_labels,
            device=str(DEVICE), chunk=128, dtype=torch.float32,
        )
        baseline_silhouette_reason = None
    else:
        baseline_silhouette = None
        baseline_silhouette_reason = "silhouette_requires_2_to_N_minus_1_used_classes"

    map_metrics[specimen] = {
        "crop": crop["bounds"], "valid_pixels": int(mask.sum()),
        "latent_lla": asdict(lla), "snv_baseline_lla": asdict(baseline_lla),
        "silhouette": silhouette, "silhouette_undefined_reason": silhouette_reason,
        "snv_baseline_silhouette": baseline_silhouette,
        "snv_baseline_silhouette_undefined_reason": baseline_silhouette_reason,
        "silhouette_pixels": len(metric_indices),
        "silhouette_seed": SEED + 4000 + int(specimen.removeprefix("sample_")),
        "silhouette_local_index_sha256": array_sha256(metric_indices),
    }
    map_path = RUN_DIR / f"{specimen}_maps.npz"
    np.savez(
        map_path, latent_label_map=label_map, snv_label_map=baseline_map, valid_mask=mask,
        silhouette_local_indices=metric_indices, features=crop_z.numpy(),
        absolute_coordinates=crop["absolute_coordinates"],
    )
    map_output_records[specimen] = {
        "file": map_path.name, "sha256": sha256_file(map_path), "shape": list(mask.shape),
    }

    brightness = np.full(mask.shape, np.nan, dtype=np.float32)
    brightness[mask] = native_rows.mean(axis=1)
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    axes[0].imshow(np.ma.masked_invalid(brightness), cmap="gray")
    axes[0].set_title("Supplied mean signal")
    axes[1].imshow(np.ma.masked_where(~mask, label_map), cmap="tab10", vmin=0, vmax=K-1)
    axes[1].set_title("ChemoMAE cluster IDs")
    axes[2].imshow(np.ma.masked_where(~mask, baseline_map), cmap="tab10", vmin=0, vmax=K-1)
    axes[2].set_title("SNV spectral cluster IDs")
    for axis in axes:
        axis.set(xlabel="Native crop column", ylabel="Native crop row")
    fig.suptitle(f"{specimen}: offset (y={crop['bounds']['y0']}, x={crop['bounds']['x0']}); no mineral-class claims")
    fig.tight_layout()
    fig.savefig(RUN_DIR / f"{specimen}_maps.png", dpi=130)
    plt.show()
    print(specimen, "occupancy:", dict(zip(lla.class_labels, lla.occupancy)))
    for window in lla.windows:
        print("window", window.window, "raw", window.raw_agreement, "corrected", window.score,
              "pairs", window.valid_pairs, "undefined", window.undefined_reasons)
    print("Bounded silhouette:", silhouette, "undefined reason:", silhouette_reason)


## 10. Optional bounded vMF example

This cell is disabled by default. It fits at most 128 training latent vectors with K=4 and 5 EM iterations, then predicts a bounded held-out feature probe with the fitted mixture. It demonstrates the public API, not a fair comparison with the full quick-path cosine fit. Bessel-function and likelihood work can involve CPU computation even with CUDA features.

In [ ]:
optional_vmf_record = {"enabled": RUN_OPTIONAL_VMF}
if RUN_OPTIONAL_VMF:
    vmf_training = train_z[:min(128, len(train_z))]
    mixture = VMFMixture(
        n_components=K, d=train_z.shape[1], device=DEVICE,
        random_state=SEED, max_iter=5, dtype=torch.float32,
    ).fit(vmf_training, chunk=64)
    heldout_probe = feature_extractor.transform(clean_loader(heldout_x[:min(64, len(heldout_x))]))
    vmf_labels = mixture.predict(heldout_probe, chunk=64).cpu().numpy()
    vmf_path = RUN_DIR / "optional_vmf.pt"
    mixture.save(str(vmf_path))
    loaded_mixture = VMFMixture.load(str(vmf_path), map_location=DEVICE)
    assert np.array_equal(vmf_labels, loaded_mixture.predict(heldout_probe, chunk=64).cpu().numpy())
    optional_vmf_record.update({
        "training_feature_rows": len(vmf_training), "heldout_probe_rows": len(heldout_probe),
        "max_iter": 5, "k": K, "saved_file": vmf_path.name,
        "sha256": sha256_file(vmf_path),
    })
    print("Optional vMF held-out probe cluster IDs:", vmf_labels)
else:
    print("Optional vMF skipped; enable RUN_OPTIONAL_VMF for this bounded example.")


## 11. Export the protocol and measured runtime record

Files are written to the runtime run directory. This cell records actual transfer sizes/times, source/version information, chosen weights, RNG state, specimen/pixel/crop manifests and hashes, fit diagnostics, per-crop metrics, and roundtrip checks. Undefined floating-point metrics are exported as JSON `null` together with explicit reasons; they are not replaced by zero.

Elapsed time is measured when you run this notebook. CUDA peak allocated bytes are recorded only when CUDA is selected. Process peak RSS, where supported, is a process-lifetime value including the notebook kernel and imports, not an isolated pipeline allocation measurement. No speed, GPU type, memory availability, or metric value is promised in advance.

In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

process_peak_rss_bytes = None
process_peak_rss_note = "Unavailable on this platform."
try:
    import resource
    peak_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if platform.system() == "Linux":
        process_peak_rss_bytes = int(peak_rss * 1024)
    elif platform.system() == "Darwin":
        process_peak_rss_bytes = int(peak_rss)
    process_peak_rss_note = "Process-lifetime high-water RSS, including imports and other kernel work."
except ImportError:
    pass
runtime_record = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "installation_elapsed_seconds": _INSTALL_ELAPSED,
    "pipeline_elapsed_seconds_excluding_install": time.perf_counter() - PIPELINE_STARTED,
    "device": str(DEVICE), "cuda_opt_in": USE_CUDA, "amp": AMP,
    "cuda_device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
    "cuda_peak_allocated_bytes": torch.cuda.max_memory_allocated(DEVICE) if DEVICE.type == "cuda" else None,
    "process_peak_rss_bytes": process_peak_rss_bytes, "process_peak_rss_note": process_peak_rss_note,
}
dataset_sources = {
    "readme": "https://github.com/EleftheriaTtl/minerals-in-the-wild/blob/main/README.md",
    "release": "https://github.com/EleftheriaTtl/minerals-in-the-wild/releases/tag/v1.0.0",
    "doi": "https://doi.org/10.5281/zenodo.21414930",
    "license": "https://creativecommons.org/licenses/by/4.0/",
    "transfers": transfer_records,
    "asset_manifest_columns": asset_manifest_columns,
    "asset_manifest_preview": asset_manifest_preview,
    "source_pin_pending": True,
}
results = {
    "protocol_sha256": protocol_sha256, "versions": versions, "runtime": runtime_record,
    "dataset_sources": dataset_sources, "optimizer_groups": optimizer_groups,
    "training_first_stage": first_stage, "training_final": training_result,
    "selected_weights_file": str(selected_path.relative_to(RUN_DIR)),
    "selected_weights_sha256": sha256_file(selected_path),
    "model_roundtrip_check": model_roundtrip_check,
    "cluster_roundtrip_check": cluster_roundtrip_check,
    "representations": {"shapes": representation_shapes, "norm_ranges": representation_norms},
    "training_feature_batches": feature_batch_files,
    "reconstruction": reconstruction_metrics, "cluster_fits": clustering_diagnostics,
    "heldout_crop_metrics": map_metrics, "map_outputs": map_output_records,
    "optional_vmf": optional_vmf_record,
    "sample_indices_sha256": sha256_file(index_manifest_path),
    "final_rng_state_sha256": sha256_file(RUN_DIR / "final_rng_state.pt"),
}
(RUN_DIR / "results.json").write_text(
    json.dumps(json_safe(results), indent=2, allow_nan=False), encoding="utf-8"
)
(RUN_DIR / "DATA_ATTRIBUTION.txt").write_text(
    "Minerals in the Wild, release v1.0.0; DOI 10.5281/zenodo.21414930.\n"
    "E. Tetoula-Tsonga, G. Arvanitakis, T. Giannakas. Data license: CC BY 4.0.\n"
    "https://creativecommons.org/licenses/by/4.0/\n"
    "https://github.com/EleftheriaTtl/minerals-in-the-wild\n"
    "Modifications in this tutorial: recorded-grid linear interpolation from 273 to 256 bands, "
    "per-spectrum SNV, bounded pixel selection/cropping, learned representations and cluster maps.\n",
    encoding="utf-8",
)
artifact_files = [
    {"file": str(path.relative_to(RUN_DIR)), "bytes": path.stat().st_size, "sha256": sha256_file(path)}
    for path in sorted(RUN_DIR.rglob("*")) if path.is_file()
]
(RUN_DIR / "artifact_manifest.json").write_text(
    json.dumps(artifact_files, indent=2, allow_nan=False), encoding="utf-8"
)
print("Runtime measurements:", json.dumps(runtime_record, indent=2))
print("Exported run:", RUN_DIR)
print("Download/copy this run directory before the temporary runtime is discarded.")
print("Interpret metrics with the saved protocol, crop masks, undefined reasons, and dataset attribution.")


## Before using this as a scientific protocol or a released Colab tutorial

- Verify fresh-Colab Run all, direct asset downloads, checksums, package installation, math rendering, runtime, and peak memory on named CPU/CUDA environments. Publish the matching APIs first and pin release/source/metadata refs.
- Verify site/context metadata and related specimens/acquisitions before claiming broader generalization. Select a representative dataset subset and specimen/group split explicitly; first-twelve-file selection is only a convenience demonstration.
- Define model/K/weight selection using training and validation only, repeat seeds, fitting budgets, perturbation strengths, evaluation regions and aggregation, then freeze conditions before held-out evaluation.
- Compare fair baselines and ablations with declared costs. A short quick-path result cannot establish superiority or chemical class meaning. XRF elemental specimen targets cannot serve as pixel mineral annotations.
- Decide full-map versus cropped-map analysis, neighborhood physical scale, specimen-level aggregation, and undefined-value policy. Preserve coordinates/masks and report occupancy with LLA rather than maximizing spatial agreement alone.
- Specify the separate LFR perturbation/repetition/RNG protocol before interpreting fragmentation; no LFR result is generated here.
- Test uninterrupted versus resumed preparation/update sequences under matching conditions. The extension example covers its own streams and single-process epoch boundaries, not arbitrary external state.

This draft leaves execution outputs empty. Completing its source files does not certify the release or rerun the completed v0.2.2 thesis experiments.